In [1]:
import pandas as pd

# Cargar películas
movies = pd.read_csv("../data/movies.dat", sep="::", engine="python", 
                     names=["movie_id", "title", "genres"], encoding="latin-1")

# Cargar ratings
ratings = pd.read_csv("../data/ratings.dat", sep="::", engine="python",
                      names=["user_id", "movie_id", "rating", "timestamp"], encoding="latin-1")

# Cargar usuarios
users = pd.read_csv("../data/users.dat", sep="::", engine="python",
                    names=["user_id", "gender", "age", "occupation", "zip"], encoding="latin-1")

print("Películas:", movies.shape)
print("Ratings:", ratings.shape)
print("Usuarios:", users.shape)

Películas: (3883, 3)
Ratings: (1000209, 4)
Usuarios: (6040, 5)


In [2]:
print(movies.head())
print()
print(ratings.head())
print()
print(users.head())

   movie_id                               title                        genres
0         1                    Toy Story (1995)   Animation|Children's|Comedy
1         2                      Jumanji (1995)  Adventure|Children's|Fantasy
2         3             Grumpier Old Men (1995)                Comedy|Romance
3         4            Waiting to Exhale (1995)                  Comedy|Drama
4         5  Father of the Bride Part II (1995)                        Comedy

   user_id  movie_id  rating  timestamp
0        1      1193       5  978300760
1        1       661       3  978302109
2        1       914       3  978301968
3        1      3408       4  978300275
4        1      2355       5  978824291

   user_id gender  age  occupation    zip
0        1      F    1          10  48067
1        2      M   56          16  70072
2        3      M   25          15  55117
3        4      M   45           7  02460
4        5      M   25          20  55455


In [3]:
print("Distribución de ratings:")
print(ratings["rating"].value_counts().sort_index())
print()
print("Rating promedio:", ratings["rating"].mean().round(2))
print("Película más rankeada:")
print(ratings["movie_id"].value_counts().head(1))

Distribución de ratings:
rating
1     56174
2    107557
3    261197
4    348971
5    226310
Name: count, dtype: int64

Rating promedio: 3.58
Película más rankeada:
movie_id
2858    3428
Name: count, dtype: int64


In [4]:
movies[movies["movie_id"] == 2858]

,movie_id,title,genres
2789,2858,American Beauty (1999),Comedy|Drama


In [5]:
top_peliculas = ratings.groupby("movie_id")["rating"].agg(["count", "mean"]).round(2)
top_peliculas = top_peliculas.merge(movies[["movie_id", "title"]], on="movie_id")
top_peliculas = top_peliculas.sort_values("count", ascending=False).head(10)
print(top_peliculas[["title", "count", "mean"]])

                                                  title  count  mean
2651                             American Beauty (1999)   3428  4.32
253           Star Wars: Episode IV - A New Hope (1977)   2991  4.45
1106  Star Wars: Episode V - The Empire Strikes Back...   2990  4.29
1120  Star Wars: Episode VI - Return of the Jedi (1983)   2883  4.02
466                                Jurassic Park (1993)   2672  3.76
1848                         Saving Private Ryan (1998)   2653  4.34
575                   Terminator 2: Judgment Day (1991)   2649  4.06
2374                                 Matrix, The (1999)   2590  4.32
1178                          Back to the Future (1985)   2583  3.99
579                    Silence of the Lambs, The (1991)   2578  4.35


In [6]:
from sklearn.model_selection import train_test_split

# Nos quedamos con usuarios que tienen mas de 50 ratings para que el modelo aprenda bien
usuarios_activos = ratings.groupby("user_id")["rating"].count()
usuarios_activos = usuarios_activos[usuarios_activos >= 50].index

ratings_filtrado = ratings[ratings["user_id"].isin(usuarios_activos)]
print(f"Ratings originales: {len(ratings)}")
print(f"Ratings filtrados: {len(ratings_filtrado)}")
print(f"Usuarios activos: {len(usuarios_activos)}")

Ratings originales: 1000209
Ratings filtrados: 943471
Usuarios activos: 4297


In [7]:
from scipy.sparse import csr_matrix

# Matriz usuario-pelicula
matriz = ratings_filtrado.pivot_table(index="user_id", columns="movie_id", values="rating")
print(f"Tamaño de la matriz: {matriz.shape}")
print(f"Densidad: {ratings_filtrado.shape[0] / (matriz.shape[0] * matriz.shape[1]) * 100:.2f}%")

Tamaño de la matriz: (4297, 3689)
Densidad: 5.95%


In [8]:
from sklearn.neighbors import NearestNeighbors

# Rellenamos los NaN con 0
matriz_sparse = csr_matrix(matriz.fillna(0).values)

# Entrenamos el modelo
modelo_knn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=10)
modelo_knn.fit(matriz_sparse)

print("Modelo entrenado OK")

Modelo entrenado OK


In [10]:
# Transponer la matriz para que las filas sean peliculas
matriz_peliculas = csr_matrix(matriz.fillna(0).T.values)

modelo_knn = NearestNeighbors(metric="cosine", algorithm="brute", n_neighbors=10)
modelo_knn.fit(matriz_peliculas)

def recomendar_peliculas(titulo, n_recomendaciones=5):
    pelicula = movies[movies["title"].str.contains(titulo, case=False, na=False)]
    
    if pelicula.empty:
        print(f"No se encontró ninguna película con '{titulo}'")
        return
    
    movie_id = pelicula.iloc[0]["movie_id"]
    titulo_encontrado = pelicula.iloc[0]["title"]
    
    if movie_id not in matriz.columns:
        print(f"'{titulo_encontrado}' no tiene suficientes ratings")
        return
    
    idx = list(matriz.columns).index(movie_id)
    vector = matriz_peliculas[idx]
    
    distancias, indices = modelo_knn.kneighbors(vector, n_neighbors=n_recomendaciones+1)
    
    print(f"Porque te gustó: {titulo_encontrado}\n")
    print("Te recomendamos:")
    for i, idx_pelicula in enumerate(indices.flatten()[1:]):
        movie_id_rec = matriz.columns[idx_pelicula]
        titulo_rec = movies[movies["movie_id"] == movie_id_rec]["title"].values[0]
        print(f"  {i+1}. {titulo_rec} (similitud: {1 - distancias.flatten()[i+1]:.2f})")

recomendar_peliculas("Matrix")

Porque te gustó: Matrix, The (1999)

Te recomendamos:
  1. Terminator 2: Judgment Day (1991) (similitud: 0.78)
  2. Total Recall (1990) (similitud: 0.73)
  3. Jurassic Park (1993) (similitud: 0.73)
  4. Star Wars: Episode V - The Empire Strikes Back (1980) (similitud: 0.73)
  5. Men in Black (1997) (similitud: 0.72)


In [11]:
recomendar_peliculas("Toy Story")

Porque te gustó: Toy Story (1995)

Te recomendamos:
  1. Toy Story 2 (1999) (similitud: 0.65)
  2. Groundhog Day (1993) (similitud: 0.63)
  3. Aladdin (1992) (similitud: 0.63)
  4. Bug's Life, A (1998) (similitud: 0.62)
  5. Back to the Future (1985) (similitud: 0.61)


In [12]:
recomendar_peliculas("Sixth Sense")

Porque te gustó: Sixth Sense, The (1999)

Te recomendamos:
  1. Silence of the Lambs, The (1991) (similitud: 0.67)
  2. American Beauty (1999) (similitud: 0.65)
  3. Matrix, The (1999) (similitud: 0.63)
  4. Fargo (1996) (similitud: 0.62)
  5. Usual Suspects, The (1995) (similitud: 0.62)
